# Chapter 6 - Ensemble Methods

## Preparation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

## 1. Why is bagging based on random sampling with replacement? Would bagging still reduce a forecast's variance if sampling were without replacement?

If we sample without replacement, and we choose the sample size to be the same as the dataset size $D$, then we are basically getting $N$ exactly same estimators, and the correlation between each estimator is 1, leading to no reduction in variance at all. Or we can sample $k < D$ data for each estimator, but then each estimator train on less data and may lead to higher bias.

## 2. Suppose that your training set is based on highly overlap labels (i.e., with  low uniqueness, as defined in Chapter 4).

### a. Does this make bagging prone to overfitting, or just ineffective? Why?

When there are many overlap labels, basically every bootstrap samples are very similar to each other, and therefore the correlation between estimators is near 1. Since $\text{Var}(\text{ensemble}) = \rho \sigma^2 + \frac{1-\rho}{m} \sigma^2$, when $\rho \approx 1$, then $\text{Var}(\text{ensemble}) \approx \sigma^2$, which is independent from the sample size, and there is no way to reduce the variance of the prediction, leading to overfitting.

### b. Is out-of-bag accuracy generally reliable in financial applications? Why?

No. Since the financial data are highly non-IID, where there are many highly overlapping labels. Therefore the in-of-bag samples and out-of-bag samples are highly similar, leading to out-of-bag accuracy inflation.

OOB accuracy is inflated because the OOB samples are not truly “unseen” — they share labels (and thus information) with the in-bag samples due to the overlapping window structure. This violates the core assumption behind OOB as a valid test set, making it an optimistically biased estimate of generalization error.

## 3. Build an ensemble of estimators, where the base estimator is a decision tree.

### a. How is this ensemble different from an RF?

In [8]:
X, y = make_classification(n_samples=1000, n_features=50, random_state=0)

bc = BaggingClassifier(DecisionTreeClassifier(), oob_score=True, random_state=1)
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=1)

bc.fit(X, y)
rf.fit(X, y)

print("OOB score of the bagging classifier:", bc.oob_score_)
print("OOB score of the random forest:", rf.oob_score_)

/Users/lucaswychan/advances-in-financial-ml-exercises/.venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:975: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/lucaswychan/advances-in-financial-ml-exercises/.venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:981: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


OOB score of the bagging classifier: 0.948
OOB score of the random forest: 0.96


### b. Using sklearn, produce a bagging classifier that behaves like an RF.  What parameters did you have to set up, and how?

In [11]:
rf_like_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        max_features="sqrt",        # RF default: sqrt(p) features per split
        max_depth=None,             # Fully grown trees (no pruning)
        min_samples_split=2,        # RF default
        min_samples_leaf=1,         # RF default
        splitter="best",            # Deterministic split given the feature subset
        random_state=42
    ),
    n_estimators=100,               # Number of trees
    max_samples=1.0,                # Bootstrap with 100% of training rows
    max_features=1.0,               # Pass ALL features to each tree (subsampling handled inside the tree via max_features)
    bootstrap=True,                 # Sample rows WITH replacement (bagging)
    bootstrap_features=False,       # Don't re-sample feature columns at the bag level
    n_jobs=-1,
    random_state=1,
    oob_score=True
)

rf_like_bagging.fit(X, y)

print("OOB score of the bagging classifier:", rf_like_bagging.oob_score_)

OOB score of the bagging classifier: 0.954


## 4. Consider the relation between an RF, the number of trees it is composed of,  and the number of features utilized:

### a. Could you envision a relation between the minimum number of trees needed in an RF and the number of features utilized?

More features in a tree --> more correlated of two trees --> less trees needed.

Less features in a tree --> trees are more diversified --> need more tree to average out the higher individual variance --> more trees needed

### b. Could the number of trees be too small for the number of features used?

Yes, and it will lead to some features that are not used, which will lead to higher variance, where the OOB error estimates are too noisy and unstable.

### c. Could the number of trees be too high for the number of observations available?

Yes, and it will lead to higher correlation between the estimators, because the estimators will look increasingly similar. Besides, it wastes the computation time.

## 5. How is out-of-bag accuracy different from stratified k-fold (with shuffling)  cross-validation accuracy?

1. Out-of-bag did not guarantee the class balance between in-of-bag and out-of-bag, where out-of-bag samples may contain so few data for a label. In contrast, the stratified k-fold guarantee that each fold must contain the same class distribution compared to the full dataset. So the accuracy may be affected by the class distribution.

2. OOB accuracy tends to be a slightly pessimistic estimate (because each tree only trains on ~63.2% of data), while stratified k-fold with k=5 uses 80% of data for training, which more closely reflects the final model’s capacity